### 📖 과제 1: LLM 기반 2단계 그래프

1차 노드에서 답변을 생성하고

2차 노드에서 뒤에 정리 문구를 추가하는 그래프를 구현하세요.

```
뒤:
---
추가 문의가 필요하시다면 챗봇 담당자(example@gmail.com)에게 연락 부탁드립니다.
```

**State**:
- answer / answer_len

**그래프 구조**:

START → generate_answer → format_answer → END

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model

# TODO: State 정의 (question, answer, answer_len)

# TODO: generate_answer 노드 구현 (LLM 호출하여 답변 생성)

# TODO: format_answer 노드 구현 (답변 뒤에 포맷 추가)

# TODO: 그래프 구성 (START → generate_answer → format_answer → END)

# TODO: 실행 및 결과 확인

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("API Key가 설정되었습니다.")

<details>

<summary>모범 답안 펼쳐보기</summary>

```python
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model

# State 정의

class InputState(TypedDict):
    question: str  # 입력 질문

class State(InputState):
    answer: Annotated[str, add]  # 문자열 연결
    answer_len: Annotated[int, add]  # 답변 길이

# generate_answer 노드: LLM으로 답변 생성
def generate_answer(state: State):
    question = state["question"]

    # LLM 호출
    model = init_chat_model("gpt-5.4-mini")
    response = model.invoke(question)
    answer = response.content

    return {"answer": answer, "answer_len": len(answer)}

# format_answer 노드: 답변 뒤 포맷팅
def format_answer(state: State):
    print("format_answer 노드에 입력된 상태:", state)
    suffix = "\n\n---\n추가 문의가 필요하시다면 챗봇 담당자(example@gmail.com)에게 연락 부탁드립니다."
    return {"answer": suffix, "answer_len": len(suffix)}

# 그래프 구성
graph_builder = StateGraph(State, input_schema=InputState, output_schema=State)
graph_builder.add_node("generate_answer", generate_answer)
graph_builder.add_node("format_answer", format_answer)

graph_builder.add_edge(START, "generate_answer")
graph_builder.add_edge("generate_answer", "format_answer")
graph_builder.add_edge("format_answer", END)

graph = graph_builder.compile()

# 실행 및 결과 확인
input = {"question": "넌 어떤 기능이 있어?"}
result = graph.invoke(input)
print("최종 결과:")
print(f"질문: {result['question']}")
print(f"답변:\n{result['answer']}")
print(f"답변 길이: {result['answer_len']} 자")
```
</details>

---